# Laboratorio 04.3 — Comparación de modelos

Diseñaremos una comparación controlada entre dos configuraciones de modelo con la misma tarea, contrato de salida y temperatura. La invocación remota es opcional y queda desactivada.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Definir una comparación con al menos dos configuraciones de modelo.
- Mantener constantes tarea, entrada y parámetros no estudiados.
- Registrar calidad verificable, cumplimiento de esquema, latencia y tokens disponibles.
- Informar valores desconocidos sin atribuir superioridad universal a un proveedor.

## 2. Conceptos

Una comparación justa fija la tarea, las entradas, la rúbrica, el prompt y los parámetros que no forman parte del experimento. Modelos distintos pueden reportar metadatos diferentes; costo requiere tarifas vigentes y no se deduce solo del conteo de tokens. Una prueba pequeña sirve para verificar el protocolo, no para establecer un ganador universal.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Pregunta y prompt fijo] --> B[Configuracion modelo A]
    A --> C[Configuracion modelo B]
    B --> D[Adaptador comun]
    C --> D
    D --> E[JSON validado]
    E --> F[Metricas por modelo]
    F --> G[DataFrame de comparacion]
```

## 4. Preparación del entorno

Configuramos pandas, Pydantic y la fábrica de modelos. Los nombres se leen de `models.yaml` y pueden sobrescribirse por entorno; ninguna clave se imprime.

In [ ]:
import os
from time import perf_counter

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from llm_engineering.environment import preparar_entorno
from llm_engineering.models import get_llm, load_model_registry

raiz_curso = preparar_entorno()
registro = load_model_registry()

## 5. Definición del problema

Clasificaremos una pregunta de soporte que tiene una etiqueta de referencia clara. El conjunto de prueba contiene una sola consulta didáctica; no es suficiente para una afirmación estadística amplia. El objetivo principal es mostrar un protocolo medible y reproducible.

## 6. Línea base

La referencia conocida es la categoría `envios`. Antes de llamar modelos, comprobamos que el contrato pueda representar la respuesta esperada y registramos que el experimento comenzará con cero solicitudes.

In [ ]:
class RespuestaComparacion(BaseModel):
    model_config = ConfigDict(extra='forbid')

    categoria: str = Field(min_length=3)
    respuesta: str = Field(min_length=8)


pregunta = 'Mi paquete todavía no ha llegado. ¿Qué debo revisar?'
categoria_esperada = 'envios'
print(RespuestaComparacion(categoria=categoria_esperada, respuesta='Revisa el seguimiento del pedido.'))

## 7. Implementación

Construimos una única instrucción y dos configuraciones diferentes: Ollama local y OpenAI configurable. El cambio controlado es el modelo/proveedor; temperatura, consulta y formato se mantienen iguales. Puedes sustituir ambos identificadores desde configuración.

In [ ]:
modelo_ollama = os.getenv('OLLAMA_MODEL', registro.providers['ollama'].model)
modelo_openai = os.getenv('OPENAI_MODEL', registro.providers['openai'].model)
configuraciones = [
    {'proveedor': 'ollama', 'modelo': modelo_ollama, 'temperatura': 0.0},
    {'proveedor': 'openai', 'modelo': modelo_openai, 'temperatura': 0.0},
]
prompt = (
    'Clasifica la consulta como envios, devoluciones, pagos o cambios.\n'
    'Devuelve solo JSON con las claves categoria y respuesta.\n'
    f'Consulta: {pregunta}'
)
tabla_configuracion = pd.DataFrame(configuraciones)
display(tabla_configuracion)
tabla_configuracion.to_csv(raiz_curso / 'outputs' / '04_03_model_configurations.csv', index=False)

## 8. Experimentos

Con el interruptor activado se realiza como máximo una llamada por configuración: dos solicitudes en total. OpenAI puede generar cargos; Ollama requiere un servidor local. Revisa proveedor, identificadores y credenciales antes de habilitar este bloque.

In [ ]:
EJECUTAR_COMPARACION = False
llamadas_maximas = len(configuraciones)
print(f'Llamadas maximas planeadas: {llamadas_maximas}; por defecto: 0')
filas = []
if EJECUTAR_COMPARACION:
    for configuracion in configuraciones:
        inicio = perf_counter()
        registro_uso = {}
        try:
            modelo = get_llm(
                configuracion['proveedor'],
                model=configuracion['modelo'],
                temperature=configuracion['temperatura'],
            )
            mensaje = modelo.invoke(prompt)
            registro_uso = getattr(mensaje, 'usage_metadata', None) or {}
            metadatos = getattr(mensaje, 'response_metadata', None) or {}
            uso_proveedor = metadatos.get('token_usage', {})
            salida = RespuestaComparacion.model_validate_json(str(mensaje.content))
            filas.append({
                **configuracion,
                'categoria': salida.categoria,
                'schema_valid': True,
                'latencia_segundos': perf_counter() - inicio,
                'tokens_entrada': registro_uso.get('input_tokens', uso_proveedor.get('prompt_tokens')),
                'tokens_salida': registro_uso.get('output_tokens', uso_proveedor.get('completion_tokens')),
                'costo_estimado': None,
                'tipo_error': None,
            })
        except Exception as error:
            filas.append({
                **configuracion,
                'categoria': None,
                'schema_valid': False,
                'latencia_segundos': perf_counter() - inicio,
                'tokens_entrada': registro_uso.get('input_tokens'),
                'tokens_salida': registro_uso.get('output_tokens'),
                'costo_estimado': None,
                'tipo_error': type(error).__name__,
            })
resultados = pd.DataFrame(filas)
display(resultados)

## 9. Evaluación

Medimos exactitud de categoría y cumplimiento de esquema, junto con latencia y tokens cuando existan. `costo_estimado` permanece desconocido porque requiere tarifas configurables y vigentes. La tabla vacía representa que no se realizaron solicitudes.

In [ ]:
if resultados.empty:
    metricas_modelos = pd.DataFrame(columns=[
        'proveedor', 'modelo', 'accuracy_categoria', 'schema_valid',
        'latencia_media_segundos', 'tokens_entrada', 'tokens_salida', 'costo_estimado',
    ])
else:
    resultados['categoria_correcta'] = resultados['categoria'] == categoria_esperada
    metricas_modelos = resultados.rename(columns={'categoria_correcta': 'accuracy_categoria'})[
        ['proveedor', 'modelo', 'categoria_correcta', 'schema_valid', 'latencia_segundos',
         'tokens_entrada', 'tokens_salida', 'costo_estimado']
    ].rename(columns={'categoria_correcta': 'accuracy_categoria', 'latencia_segundos': 'latencia_media_segundos'})
display(metricas_modelos)
if not resultados.empty:
    resultados.to_csv(raiz_curso / 'outputs' / '04_03_model_results.csv', index=False)
metricas_modelos.to_csv(raiz_curso / 'outputs' / '04_03_model_metrics.csv', index=False)

## 10. Discusión

Con una sola consulta, una diferencia de resultado es una observación, no una conclusión general. Repite sobre consultas representativas, revisa errores y documenta versión de prompt, modelos y fecha. Los metadatos y precios cambian por proveedor; no rellenes métricas ausentes con cero.

## 11. Ejercicios

1. Cambia la consulta manteniendo fijo el resto y añade una segunda etiqueta de referencia.
2. Compara dos modelos locales con idéntico prompt.
3. Añade una rúbrica humana para calidad y registra desacuerdos con exact match.
4. Implementa una tabla de tarifas versionada y estima costo solo cuando haya tokens disponibles.

## 12. Desafío

Diseña un benchmark con al menos 30 consultas, semilla fija para selección y un límite de llamadas por modelo. Reporta intervalos de confianza, fallos, tokens y costo estimado, sin declarar un proveedor universalmente superior.

## 13. Ideas clave

- Compara modelos bajo las mismas condiciones.
- Mide calidad, formato, latencia y uso por separado.
- Estima costos solo con tarifas y tokens verificables.
- Una muestra pequeña no establece superioridad universal.